[<img src="../quantumsymmetry_logo.png" alt="QuantumSymmetry" width="450"/>](https://github.com/dariopicozzi/quantumsymmetry)

> **Note:** if you are running this notebook on Google Colab, the next cell will install `quantumsymmetry` and its dependencies (this might take a few minutes):

In [1]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip -q install quantumsymmetry
    !pip -q install pylatexenc

# Natural-gradient VQE for molecules

This is the first application from the paper: a symmetry-adapted variational quantum eigensolver (VQE). It is also where the two halves of `quantumsymmetry` meet. The [first tutorial series](01_welcome.ipynb) builds a **symmetry-adapted encoding** (SAE) that tapers a molecular Hamiltonian onto the smallest possible number of qubits; this series builds the smallest **universal tree ansatz** on exactly the basis states that the target symmetry sector allows. Put together, they give a tiny, exact VQE.

We run it on lithium hydride (LiH) in a (4 electrons, 4 orbitals) complete active space.

## The symmetry-adapted Hamiltonian

`Encoding` runs the PySCF mean-field calculation, picks up the molecule's symmetries (particle number, spin projection and the $C_{2v}$ point-group irrep) and tapers the Hamiltonian. The (4, 4) active space means 4 electrons in 4 spatial orbitals, i.e. 8 spin-orbitals (8 qubits) before tapering; the three symmetries above remove three of them, so LiH lands on just **5 qubits**.

In [2]:
import numpy as np
from quantumsymmetry import Encoding, MinimalCircuit, minimize_energy

enc = Encoding('Li 0 0 0; H 0 0 1.5957', 'sto-3g', CAS=(4, 4),
               output_format='qiskit')
H = enc.hamiltonian

exact = float(np.linalg.eigvalsh(H.to_matrix())[0])
print('encoded qubits :', enc.encoded_qubits)
print('exact energy   :', round(exact, 10), 'Ha')

encoded qubits : 5
exact energy   : -7.8631977779 Ha


## The symmetry-adapted support

Symmetry does more than shrink the qubit count: within those 5 qubits, only a handful of the $2^5 = 32$ computational-basis states carry the right particle number and spin. `symmetry_adapted_support` lists them, and they are exactly the *leaves* the tree ansatz should keep. `hartree_fock_leaf` gives the encoded Hartree-Fock determinant, the natural single-determinant starting point.

In [3]:
support = enc.symmetry_adapted_support()
hf_leaf = enc.hartree_fock_leaf

print('symmetry-adapted leaves :', len(support), 'of', 2 ** enc.encoded_qubits)
print('Hartree-Fock leaf       :', hf_leaf)

symmetry-adapted leaves : 18 of 32
Hartree-Fock leaf       : 3


## The pruned ansatz

We hand the support straight to `MinimalCircuit`: the tree is pruned to span only those leaves. `minimize_energy` then runs natural-gradient descent from the Hartree-Fock leaf, taking the exact shift-rule gradient and dividing by the diagonal metric at each step. No auxiliary metric circuits, no regularisation.

In [4]:
mc = MinimalCircuit(enc.encoded_qubits, support=support)
result = minimize_energy(mc, H, reference_basis_index=hf_leaf)

print('VQE energy   :', round(result.energy, 10), 'Ha')
print('exact energy :', round(exact, 10), 'Ha')
print('error        :', f'{abs(result.energy - exact):.2e}', 'Ha')
print('parameters   :', mc.num_parameters)
print('CNOTs        :', mc.circuit.count_ops().get('cx', 0))

VQE energy   : -7.8631831042 Ha
exact energy : -7.8631977779 Ha
error        : 1.47e-05 Ha
parameters   : 17
CNOTs        : 28


The pruned ansatz reaches chemical accuracy on a 5-qubit active space with 17 parameters and 28 CNOTs. The same recipe scales to the bigger molecules of the paper (BeH$_2$, H$_2$O, NH$_3$): the symmetry-adapted encoding fixes the qubit count and the active leaves, and the tree compiler builds the smallest universal ansatz on them, giving CNOT counts one to two orders of magnitude below UCCSD.

### The series

1. <a href="tree_01_welcome.ipynb" />Welcome: the binary-tree ansatz</a>
2. <a href="tree_02_ansatz_and_metric.ipynb" />The ansatz, its pruning and its diagonal metric</a>
3. <a href="tree_03_vqe.ipynb" />Natural-gradient VQE for molecules</a>
4. <a href="tree_04_dynamics.ipynb" />Real-time evolution</a>
5. <a href="tree_05_sampling.ipynb" />Sector-Haar sampling</a>
6. <a href="tree_06_dressing.ipynb" />Dressing the ansatz: a symmetry-adapted Schrieffer-Wolff transformation</a>
7. <a href="tree_07_spin.ipynb" />Spin adaptation: exact total spin on the tree</a>

<p style="text-align: left"> <a href="tree_02_ansatz_and_metric.ipynb" />&lt; Previous: The ansatz, its pruning and its diagonal metric</a> </p>
<p style="text-align: right"> <a href="tree_04_dynamics.ipynb" />Next: Real-time evolution &gt;</a> </p>